In [1]:
%load_ext autoreload
%autoreload 2

from config import SimConfig, HybridSimConfig
from src.plants.fc_only_plant import FuelCellOnlyPlant
from src.plants.hybrid_plant import FuelCellBatteryPlant
from src.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker
from src.plotting import plot_dashboard

# Initialize Configs and Plants
cfg_base = SimConfig()
plant_base = FuelCellOnlyPlant(cfg_base)

cfg_hybrid = HybridSimConfig(lambda_scale=60, dt=5.0)
plant_hybrid = FuelCellBatteryPlant(cfg_hybrid)

# Cache data to RAM once
fleet_cache = load_and_cache_entire_fleet(cfg_base)

# Initialize our new Orchestrator (Excluding bad sensor days 1, 2, and 3)
benchmarker = VoyageBenchmarker(fleet_cache, exclude_days=[1, 2, 3])

Beginning memory staging of all 14 fleet files into RAM...
 -> Day 01 successfully cached in RAM.
 -> Day 02 successfully cached in RAM.
 -> Day 03 successfully cached in RAM.
 -> Day 04 successfully cached in RAM.
 -> Day 05 successfully cached in RAM.
 -> Day 06 successfully cached in RAM.
 -> Day 07 successfully cached in RAM.
 -> Day 08 successfully cached in RAM.
 -> Day 09 successfully cached in RAM.
 -> Day 10 successfully cached in RAM.
 -> Day 11 successfully cached in RAM.
 -> Day 12 successfully cached in RAM.
 -> Day 13 successfully cached in RAM.
 -> Day 14 successfully cached in RAM.

All 14 operational days securely held in RAM. Disk I/O locked.


In [2]:
# Create declarative definitions for whatever you want to test
baseline_heuristic = {
    "name": "Baseline Heuristic", "is_hybrid": False, "strategy": "HEURISTIC",
    "config": cfg_base, "plant": plant_base
}

baseline_sdp = {
    "name": "Baseline SDP", "is_hybrid": False, "strategy": "SDP",
    "config": cfg_base, "plant": plant_base
}

hybrid_heuristic = {
    "name": "Hybrid Heuristic", "is_hybrid": True, "strategy": "HEURISTIC",
    "config": cfg_hybrid, "plant": plant_hybrid
}

hybrid_tensor_sdp = {
    "name": "Hybrid Tensor SDP", "is_hybrid": True, "strategy": "SDP",
    "sdp_variant": "TENSOR_SWEEP", "config": cfg_hybrid, "plant": plant_hybrid
}

hybrid_mean_sdp = {
    "name": "Hybrid Mean SDP", "is_hybrid": True, "strategy": "SDP",
    "sdp_variant": "MEAN_PROXY", "config": cfg_hybrid, "plant": plant_hybrid
}

In [3]:
# Compare multiple strategies head-to-head
approaches_to_compare = {
    "Baseline Heuristic": baseline_heuristic,
    "Baseline Optimized": baseline_sdp,
    "Hybrid Heuristic": hybrid_heuristic,
    "Hybrid Optimized (Tensor)": hybrid_tensor_sdp,
    "Hybrid Optimized (Mean)": hybrid_mean_sdp
}

# Train on Days 4 through 13, Test on Day 14
df_comparison, sims = benchmarker.compare_approaches(
    approaches_to_compare, 
    train_days=[4, 5, 6, 7, 8, 9, 10, 11, 12, 13], 
    test_day=14
)

display(df_comparison)


Comparing 5 approaches | Train: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13] | Test: Day 14
 -> Running: Baseline Heuristic
 -> Running: Baseline Optimized
 -> Running: Hybrid Heuristic
 -> Running: Hybrid Optimized (Tensor)
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (50 paths)...
 -> Tensors Cached. Initiating O(M^2) Online Sweep...
 -> Running: Hybrid Optimized (Mean)
 -> Launching MEAN_PROXY Solver...


,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s)
Baseline Heuristic,104.656666,34.656666,70.0,0.000000,NaN,0.262363
Baseline Optimized,124.768658,73.768658,51.0,0.000000,NaN,0.272738
Hybrid Heuristic,412.864901,246.722552,73.0,93.142349,49.558615,0.052630
Hybrid Optimized (Tensor),255.609916,68.381239,79.0,108.228677,52.749714,39.378638
Hybrid Optimized (Mean),203.042776,30.614499,66.0,106.428277,52.755311,78.927583


In [4]:
# Plot the Baseline SDP
# plot_dashboard(sims['Baseline Heuristic'], 'Baseline Heuristic', test_day=14, layout='grid')
# plot_dashboard(sims['Baseline Optimized'], 'Baseline Optimized', test_day=14, layout='grid')

# Plot the Hybrid comparison dashboard for the best performing approach
# plot_dashboard(sims["Hybrid Heuristic"], "Hybrid Heuristic", test_day=14, layout='grid')
plot_dashboard(sims["Hybrid Optimized (Mean)"], "Hybrid Optimized (Mean)", test_day=14, layout='grid')
plot_dashboard(sims["Hybrid Optimized (Tensor)"], "Hybrid Optimized (Tensor)", test_day=14, layout='grid')


In [5]:
# Run Chronological Forward Chaining for the Hybrid Mean SDP
# print("--- APPROACH B: CHRONOLOGICAL FORWARD CHAINING ---")
# df_forward = benchmarker.run_forward_chaining(hybrid_mean_sdp, min_train_days=1)
# display(df_forward)

# Run Leave-One-Out for the Baseline SDP
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo_mean = benchmarker.run_leave_one_out(hybrid_mean_sdp)
display(df_loo_mean)
df_loo_tensor = benchmarker.run_leave_one_out(hybrid_tensor_sdp)
display(df_loo_tensor)


--- APPROACH A: LEAVE ONE OUT ---

Starting Leave-One-Out CV for: Hybrid Mean SDP
 -> LOO Fold: Testing on 4 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 5 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 6 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 7 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 8 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 9 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 10 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 11 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 12 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 13 | Training on remaining 10 days
 -> Laun

,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s),simulator
Test Day 04,306.201412,81.960148,88.0,136.241263,51.084721,74.822102,<src.simulator.HybridSimulator object at 0x000...
Test Day 05,108.199547,2.398598,30.0,75.800949,58.608416,77.826051,<src.simulator.HybridSimulator object at 0x000...
Test Day 06,230.824076,43.129132,89.0,98.694943,53.193770,91.119165,<src.simulator.HybridSimulator object at 0x000...
Test Day 07,217.051077,28.981553,87.0,101.069525,52.253627,83.149222,<src.simulator.HybridSimulator object at 0x000...
Test Day 08,219.776151,44.279772,92.0,83.496379,51.442193,85.951355,<src.simulator.HybridSimulator object at 0x000...
Test Day 09,203.406743,16.332434,73.0,114.074308,54.084824,82.773329,<src.simulator.HybridSimulator object at 0x000...
Test Day 10,181.086586,27.116983,57.0,96.969603,51.411074,80.815094,<src.simulator.HybridSimulator object at 0x000...
Test Day 11,167.508260,6.019545,49.0,112.488715,53.588149,80.007338,<src.simulator.HybridSimulator object at 0x000...
Test Day 12,198.743231,17.661907,87.0,94.081324,53.049061,74.770232,<src.simulator.HybridSimulator object at 0x000...
Test Day 13,225.929126,33.349706,95.0,97.579420,52.779262,79.592962,<src.simulator.HybridSimulator object at 0x000...



Starting Leave-One-Out CV for: Hybrid Tensor SDP
 -> LOO Fold: Testing on 4 | Training on remaining 10 days
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (50 paths)...
 -> Tensors Cached. Initiating O(M^2) Online Sweep...
 -> LOO Fold: Testing on 5 | Training on remaining 10 days
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (50 paths)...
 -> Tensors Cached. Initiating O(M^2) Online Sweep...
 -> LOO Fold: Testing on 6 | Training on remaining 10 days
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (50 paths)...
 -> Tensors Cached. Initiating O(M^2) Online Sweep...
 -> LOO Fold: Testing on 7 | Training on remaining 10 days
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (50 paths)...
 -> Tensors Cached. Initiating O(M^2) Online Sweep...
 -> LOO Fold: Testing on 8 | Training on remaining 10 days
 -> Launching TENSOR_SWEEP Solver...
 -

,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s),simulator
Test Day 04,303.314699,93.329356,89.0,120.985343,50.569080,34.648252,<src.simulator.HybridSimulator object at 0x000...
Test Day 05,132.865682,30.517648,27.0,75.348034,56.609327,35.115518,<src.simulator.HybridSimulator object at 0x000...
Test Day 06,302.311498,109.684509,104.0,88.626989,53.599161,31.915164,<src.simulator.HybridSimulator object at 0x000...
Test Day 07,328.623674,126.485574,94.0,108.138100,52.186248,35.001533,<src.simulator.HybridSimulator object at 0x000...
Test Day 08,213.204941,48.869899,78.0,86.335041,51.025337,32.143038,<src.simulator.HybridSimulator object at 0x000...
Test Day 09,361.968677,168.476917,88.0,105.491760,53.022386,35.010378,<src.simulator.HybridSimulator object at 0x000...
Test Day 10,207.972437,35.625981,83.0,89.346456,51.222161,41.769905,<src.simulator.HybridSimulator object at 0x000...
Test Day 11,228.352231,47.714158,64.0,116.638073,53.843829,45.359570,<src.simulator.HybridSimulator object at 0x000...
Test Day 12,256.750123,53.735114,111.0,92.015009,54.145763,40.281806,<src.simulator.HybridSimulator object at 0x000...
Test Day 13,241.069137,55.426016,83.0,102.643121,52.777301,35.326854,<src.simulator.HybridSimulator object at 0x000...


In [6]:
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo_baseline_sdp = benchmarker.run_leave_one_out(baseline_sdp)
display(df_loo_baseline_sdp)
df_loo_baseline_heuristic = benchmarker.run_leave_one_out(baseline_heuristic)
display(df_loo_baseline_heuristic)


--- APPROACH A: LEAVE ONE OUT ---

Starting Leave-One-Out CV for: Baseline SDP
 -> LOO Fold: Testing on 4 | Training on remaining 10 days
 -> LOO Fold: Testing on 5 | Training on remaining 10 days
 -> LOO Fold: Testing on 6 | Training on remaining 10 days
 -> LOO Fold: Testing on 7 | Training on remaining 10 days
 -> LOO Fold: Testing on 8 | Training on remaining 10 days
 -> LOO Fold: Testing on 9 | Training on remaining 10 days
 -> LOO Fold: Testing on 10 | Training on remaining 10 days
 -> LOO Fold: Testing on 11 | Training on remaining 10 days
 -> LOO Fold: Testing on 12 | Training on remaining 10 days
 -> LOO Fold: Testing on 13 | Training on remaining 10 days
 -> LOO Fold: Testing on 14 | Training on remaining 10 days


,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s),simulator
Test Day 04,150.503124,106.503124,44.0,0.0,NaN,0.030669,<src.simulator.Simulator object at 0x0000023D8...
Test Day 05,59.235491,50.235491,9.0,0.0,NaN,0.030785,<src.simulator.Simulator object at 0x0000023D8...
Test Day 06,133.406407,83.406407,50.0,0.0,NaN,0.031557,<src.simulator.Simulator object at 0x0000023D8...
Test Day 07,138.855547,82.855547,56.0,0.0,NaN,0.032078,<src.simulator.Simulator object at 0x0000023D8...
Test Day 08,96.728959,57.728959,39.0,0.0,NaN,0.030930,<src.simulator.Simulator object at 0x0000023D8...
Test Day 09,104.754120,75.754120,29.0,0.0,NaN,0.035611,<src.simulator.Simulator object at 0x0000023D8...
Test Day 10,82.377420,59.377420,23.0,0.0,NaN,0.029833,<src.simulator.Simulator object at 0x0000023D8...
Test Day 11,109.374494,77.374494,32.0,0.0,NaN,0.033886,<src.simulator.Simulator object at 0x0000023D8...
Test Day 12,132.107850,76.107850,56.0,0.0,NaN,0.028537,<src.simulator.Simulator object at 0x0000023D8...
Test Day 13,140.048920,78.048920,62.0,0.0,NaN,0.026542,<src.simulator.Simulator object at 0x0000023D8...



Starting Leave-One-Out CV for: Baseline Heuristic
 -> LOO Fold: Testing on 4 | Training on remaining 10 days
 -> LOO Fold: Testing on 5 | Training on remaining 10 days
 -> LOO Fold: Testing on 6 | Training on remaining 10 days
 -> LOO Fold: Testing on 7 | Training on remaining 10 days
 -> LOO Fold: Testing on 8 | Training on remaining 10 days
 -> LOO Fold: Testing on 9 | Training on remaining 10 days
 -> LOO Fold: Testing on 10 | Training on remaining 10 days
 -> LOO Fold: Testing on 11 | Training on remaining 10 days
 -> LOO Fold: Testing on 12 | Training on remaining 10 days
 -> LOO Fold: Testing on 13 | Training on remaining 10 days
 -> LOO Fold: Testing on 14 | Training on remaining 10 days


,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s),simulator
Test Day 04,118.017727,39.017727,79.0,0.0,NaN,0.003176,<src.simulator.Simulator object at 0x0000023D8...
Test Day 05,38.943922,19.943922,19.0,0.0,NaN,0.001507,<src.simulator.Simulator object at 0x0000023D8...
Test Day 06,108.843847,34.843847,74.0,0.0,NaN,0.002001,<src.simulator.Simulator object at 0x0000023D8...
Test Day 07,105.954946,33.954946,72.0,0.0,NaN,0.003427,<src.simulator.Simulator object at 0x0000023D8...
Test Day 08,95.270660,26.270660,69.0,0.0,NaN,0.002515,<src.simulator.Simulator object at 0x0000023D8...
Test Day 09,97.084605,32.084605,65.0,0.0,NaN,0.003517,<src.simulator.Simulator object at 0x0000023D8...
Test Day 10,76.024365,25.024365,51.0,0.0,NaN,0.001999,<src.simulator.Simulator object at 0x0000023D8...
Test Day 11,95.690761,31.690761,64.0,0.0,NaN,0.003005,<src.simulator.Simulator object at 0x0000023D8...
Test Day 12,105.612260,33.612260,72.0,0.0,NaN,0.002504,<src.simulator.Simulator object at 0x0000023D8...
Test Day 13,130.357565,44.357565,86.0,0.0,NaN,0.003057,<src.simulator.Simulator object at 0x0000023D8...
